# Extended ML Tasks


In [ ]:
%pip install imbalanced-learn -q

## Imports & Setup


In [ ]:
# ============================================================
# COLAB SETUP — Run this cell first every session
# ============================================================
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, shutil
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Base path
BASE = '/content/drive/MyDrive/nasa_turbofan_project'

# Make utils importable
os.makedirs('/content/utils', exist_ok=True)
shutil.copy(f'{BASE}/utils/nasa_score.py', '/content/utils/nasa_score.py')
sys.path.append('/content')
from utils.nasa_score import nasa_score

# Initialize metrics list
all_metrics = []

print("Setup complete!")

## Phase 1: Binary Label


In [ ]:
# Load processed data
train_df = pd.read_csv(f'{BASE}/data/processed_train.csv')
test_df = pd.read_csv(f'{BASE}/data/processed_test.csv')

# Handle labels if not present
if 'will_fail_30' not in train_df.columns:
    if 'RUL' in train_df.columns:
        train_df['will_fail_30'] = (train_df['RUL'] <= 30).astype(int)
    else:
        raise KeyError("RUL column missing from train_df")

# Load correct test properties
rul_test = pd.read_csv(f'{BASE}/data/RUL_FD001.txt', sep=r'\s+', header=None, names=['RUL'])
y_test_class = (rul_test['RUL'] <= 30).astype(int)
y_test_rul = rul_test['RUL']

# Separate features from targets
test_features = test_df.groupby('unit').last().reset_index()

exclude_cols = ['unit', 'cycle', 'RUL', 'will_fail_30', 'op1', 'op2', 'op3']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

X_train = train_df[feature_cols]
y_train_rul = train_df['RUL']
y_train_class = train_df['will_fail_30']

X_test = test_features[feature_cols]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

## Phase 2: KNN Regressor


In [ ]:
knn_reg = KNeighborsRegressor()
param_grid_knn = {'n_neighbors': [3, 5, 7, 10, 15]}
grid_knn = GridSearchCV(knn_reg, param_grid_knn, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_knn.fit(X_train, y_train_rul)

print(f"Best KNN parameters: {grid_knn.best_params_}")

# Evaluate
knn_preds = grid_knn.predict(X_test)
rmse_knn = np.sqrt(mean_squared_error(y_test_rul, knn_preds))
mae_knn = mean_absolute_error(y_test_rul, knn_preds)
nasa_knn = nasa_score(y_test_rul.values, knn_preds)

print(f"KNN RMSE: {rmse_knn:.2f}, MAE: {mae_knn:.2f}, NASA Score: {nasa_knn:.2f}")

all_metrics.append({
    'Model': 'KNN Regressor',
    'Task': 'Regression',
    'RMSE': rmse_knn,
    'MAE': mae_knn,
    'NASA_Score': nasa_knn,
    'Accuracy': np.nan, 'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan
})

## Phase 3: Support Vector Regression (SVR)


In [ ]:
# Sample 5000 rows for SVR
train_df_sample = train_df.sample(n=5000, random_state=42)
X_train_svr = train_df_sample[feature_cols]
y_train_svr = train_df_sample['RUL']

svr = SVR()
param_grid_svr = {
    'C': [0.1, 1, 10],
    'epsilon': [0.1, 0.5],
    'kernel': ['rbf', 'linear']
}
grid_svr = GridSearchCV(svr, param_grid_svr, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_svr.fit(X_train_svr, y_train_svr)

print(f"Best SVR parameters: {grid_svr.best_params_}")

# Evaluate
svr_preds = grid_svr.predict(X_test)
rmse_svr = np.sqrt(mean_squared_error(y_test_rul, svr_preds))
mae_svr = mean_absolute_error(y_test_rul, svr_preds)
nasa_svr = nasa_score(y_test_rul.values, svr_preds)

print(f"SVR RMSE: {rmse_svr:.2f}, MAE: {mae_svr:.2f}, NASA Score: {nasa_svr:.2f}")

all_metrics.append({
    'Model': 'SVR',
    'Task': 'Regression',
    'RMSE': rmse_svr,
    'MAE': mae_svr,
    'NASA_Score': nasa_svr,
    'Accuracy': np.nan, 'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan
})

## Phase 4: Binary Classification (all 4 models)


In [ ]:
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
knn_clf = KNeighborsClassifier(n_neighbors=5)
svc_clf = SVC(probability=True, random_state=42)

models_clf = {
    'Random Forest Classifier': rf_clf,
    'XGBoost Classifier': xgb_clf,
    'KNN Classifier': knn_clf,
    'SVC': svc_clf
}

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for i, (name, model) in enumerate(models_clf.items()):
    # Fit and Predict
    model.fit(X_train, y_train_class)
    preds = model.predict(X_test)
    
    # Calculate Metrics
    acc = accuracy_score(y_test_class, preds)
    prec = precision_score(y_test_class, preds)
    rec = recall_score(y_test_class, preds)
    f1 = f1_score(y_test_class, preds)
    
    # Save Metrics
    all_metrics.append({
        'Model': name,
        'Task': 'Classification',
        'RMSE': np.nan, 'MAE': np.nan, 'NASA_Score': np.nan,
        'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1
    })
    
    # Plot CM
    cm = confusion_matrix(y_test_class, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f"{name}\nF1: {f1:.2f}")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

plt.tight_layout()
plt.show()

## Phase 5: SMOTE Simulation


In [ ]:
# Simulate 10% minority ratio
class_0 = train_df[train_df['will_fail_30'] == 0]
class_1 = train_df[train_df['will_fail_30'] == 1]

# calculate what N should be for 10%
# N1 / (N0 + N1) = 0.1 -> N1 = 0.1*(N0 + N1) -> 0.9*N1 = 0.1*N0 -> N1 = N0/9
target_minority_size = int(len(class_0) / 9)
class_1_downsampled = class_1.sample(n=target_minority_size, random_state=42)

train_df_imbalanced = pd.concat([class_0, class_1_downsampled]).sample(frac=1, random_state=42)
X_train_imb = train_df_imbalanced[feature_cols]
y_train_imb = train_df_imbalanced['will_fail_30']

# Baseline RF on Imbalanced Data
rf_imb = RandomForestClassifier(n_estimators=100, random_state=42)
rf_imb.fit(X_train_imb, y_train_imb)
preds_imb = rf_imb.predict(X_test)
f1_imb_before = f1_score(y_test_class, preds_imb)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train_imb, y_train_imb)

# Trained on SMOTE data
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_smote, y_smote)
preds_smote = rf_smote.predict(X_test)
f1_imb_after = f1_score(y_test_class, preds_smote)

print(f"F1 Score BEFORE SMOTE: {f1_imb_before:.2f}")
print(f"F1 Score AFTER SMOTE: {f1_imb_after:.2f}")

all_metrics.append({
    'Model': 'Random Forest (Imbalanced Baseline)',
    'Task': 'Classification',
    'RMSE': np.nan, 'MAE': np.nan, 'NASA_Score': np.nan,
    'Accuracy': accuracy_score(y_test_class, preds_imb),
    'Precision': precision_score(y_test_class, preds_imb),
    'Recall': recall_score(y_test_class, preds_imb),
    'F1': f1_imb_before
})

all_metrics.append({
    'Model': 'Random Forest (SMOTE)',
    'Task': 'Classification',
    'RMSE': np.nan, 'MAE': np.nan, 'NASA_Score': np.nan,
    'Accuracy': accuracy_score(y_test_class, preds_smote),
    'Precision': precision_score(y_test_class, preds_smote),
    'Recall': recall_score(y_test_class, preds_smote),
    'F1': f1_imb_after
})

## Phase 6: Maintenance Thresholding


In [ ]:
# Phase 6: Maintenance Thresholding
# Extract feature importances
rf_model = models_clf['Random Forest Classifier']
top3_features = pd.Series(
    rf_model.feature_importances_, 
    index=feature_cols
).nlargest(3).index.tolist()
print(f"Top 3 Data-Driven Features identified for thresholding: {top3_features}")
f1, f2, f3 = top3_features

# Determine direction of degradation (does feature increase or decrease as failure approaches?)
f1_corr = np.corrcoef(X_test[f1], y_test_class)[0,1]
f2_corr = np.corrcoef(X_test[f2], y_test_class)[0,1]
f3_corr = np.corrcoef(X_test[f3], y_test_class)[0,1]

def get_mask(feature, threshold, corr):
    return X_test[feature] >= threshold if corr > 0 else X_test[feature] <= threshold

def get_rule_str(feature, threshold, corr):
    return f"{feature} >= {threshold:.2f}" if corr > 0 else f"{feature} <= {threshold:.2f}"

percentiles = range(10, 100, 10)
best_prob = 0
best_rule = ''
found_high_risk = False
results = []

# Evaluate combinations 
for p1 in percentiles:
    for p2 in percentiles:
        for p3 in percentiles:
            t1 = np.percentile(X_test[f1], p1)
            t2 = np.percentile(X_test[f2], p2)
            t3 = np.percentile(X_test[f3], p3)
            mask = get_mask(f1, t1, f1_corr) & get_mask(f2, t2, f2_corr) & get_mask(f3, t3, f3_corr)
            subset = y_test_class[mask]
            if len(subset) > 0:
                prob = subset.mean() * 100
                rule_str = f"{get_rule_str(f1, t1, f1_corr)} AND {get_rule_str(f2, t2, f2_corr)} AND {get_rule_str(f3, t3, f3_corr)}"
                if prob > 50 and not any(r[1] == rule_str for r in results):
                    results.append((prob, rule_str))
                if prob > best_prob:
                    best_prob = prob
                    best_rule = rule_str

print("-" * 50)
print("THRESHOLD DISCOVERY (>50% probability):")
print("-" * 50)
results.sort(key=lambda x: x[0], reverse=True)

if not results:
    print(f"No combinations > 50%. Highest found: If {best_rule} -> {best_prob:.1f}% failure probability")
else:
    for prob, rule in results:
        tag = " [HIGH RISK]" if prob >= 85 else ""
        if prob >= 85: found_high_risk = True
        print(f"If {rule} -> {prob:.1f}% failure probability{tag}")

if not found_high_risk and best_prob > 0:
    print("\nNo rule exceeded 85% probability. Here is the highest found:")
    print(f"If {best_rule} -> {best_prob:.1f}% failure probability")

## Phase 7: PCA Visualizations


In [ ]:
# PCA
pca = PCA(n_components=2, random_state=42)
X_test_pca = pca.fit_transform(X_test)

# KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_test)

# Setup DataFrames for plotting
pca_df = pd.DataFrame(X_test_pca, columns=['PC1', 'PC2'])
pca_df['Failure State'] = y_test_class.map({0: 'Healthy', 1: 'Failing'})
pca_df['Cluster'] = clusters

# Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Ground Truth
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Failure State', palette={'Healthy': 'blue', 'Failing': 'red'}, alpha=0.7, ax=axes[0])
axes[0].set_title("PCA: Actual Failure States")

# Plot 2: KMeans Clusters
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Cluster', palette='Set1', alpha=0.7, ax=axes[1])
axes[1].set_title("PCA: K-Means Clusters")

plt.tight_layout()
plt.show()

## Phase 8: Metrics Output


In [ ]:
metrics_df = pd.DataFrame(all_metrics)

# Reorder columns as requested
cols = ['Model', 'Task', 'RMSE', 'MAE', 'NASA_Score', 'Accuracy', 'Precision', 'Recall', 'F1']
metrics_df = metrics_df[cols]

metrics_df.to_csv(f'{BASE}/data/extended_ml_metrics.csv', index=False)
print("Metrics saved to Google Drive!")
display(metrics_df)